# SSF2 RL — Exploration & Data Collection

This notebook connects to the instrumented SSF2 build, lets you poke at the
observation/action space, drive the character with scripted inputs, and record
trajectories you can later use for supervised learning (behavioral cloning) or
to sanity-check your own RL implementations.

**Prerequisites** (run once, from the repo root):
```bash
cd /Users/cachemiss/Documents/projects/reflash2-fork/reflash2
.venv/bin/pip install -e python          # makes ssf2_rl importable
```

**The game auto-launches.** `env.reset()` starts the game itself if it isn't
already running — no manual terminal step needed. Requires `AIR_SDK_HOME` set,
or an AIR SDK at `~/Developer/AIRSDK*`. Game output is logged to
`.macos/adl.log`; `ssf2_rl.stop_game()` quits a game started this way. To
start it manually instead:
```bash
AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh
```

**One env, one hierarchy.** Every slot is declared with a single class
(`Agent` / `Human` / `CPU` / bots), and everything runs through `env`:
`env.reset(players=..., stage=...)` → `env.run(frames)` for real-time
evaluation → `env.step()` to step through one frame at a time.

Select the repo's `.venv` as the notebook kernel (Cmd+Shift+P → "Notebook: Select Kernel" → `.venv`).

In [7]:
#   ZeroBot / FollowBot / ScriptedBot / PolicyBot — Python-driven bots.
# Every bot inherits a safe default of mask 0 (noop), so a taken-over slot
# never silently reverts to the in-game CPU.
# (PolicyBot is also available from ssf2_rl for neural-net policies.)
from ssf2_rl.bots import Agent, ZeroBot, FollowBot, ScriptedBot, ZeroBot
from ssf2_rl.players import CPU, Human, Character, Stage
from ssf2_rl import NOOP, LEFT, RIGHT, DOWN, SPECIAL, ATTACK   # controls bit constants

from ssf2_rl.env import SSF2Env

env = SSF2Env()
step_env = SSF2Env(lockstep=True)

## 1. Connect & reset (programmatic match setup)

`reset()` restarts the match in-game and takes over the bot slots. Every slot
is declared with one class — the declaration IS the controller:

```python
env.reset(players={1: Agent("marth"), 2: CPU("samus", level=0)}, stage="battlefield")
```

- `Agent` — driven by `env.step(action)` (the RL path)
- `Human` — you play it in the game window
- `CPU` — the in-game AI at a level
- `ZeroBot` / `FollowBot` / `ScriptedBot` / `PolicyBot` — Python bots

`env.describe_matchup()` prints exactly who controls each slot. After reset,
use `env.run(frames)` to watch it play in real time, or loop `env.step()` to
step through one frame at a time.

In [ ]:

# Programmatic match setup: declare every slot + the stage right in reset().
env.close()
obs, info = env.reset(
    players={1: Agent(Character.Marth), 2: ZeroBot(Character.ZeroSuitSamus)},
    stage=Stage.bf,

)
print(env.describe_matchup())
print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

# I think initially it starts a marth vs samus game and then changes
# into marth vs zss. I need to fix that
# Zss is acting as a lvl 9 cpu, not a zerobot.

[ssf2_rl] Launching SSF2 via /Users/cachemiss/Developer/AIRSDK_51.3.3/bin/adl (log: /Users/cachemiss/Documents/projects/reflash2-fork/reflash2/.macos/adl.log) ...
[ssf2_rl] Game is up; bridge listening on 127.0.0.1:4567.
stage: battlefield
P1: external agent (step-driven), character=marth
P2: Python ZeroBot (zero), character=zamus

frame: 2 | me: Marth | opp: Zero Suit Samus


In [ ]:
# human vs cpu

obs, info = env.reset(
    players={1: Human("marth"), 2: CPU("samus", level=9)},
)
print(env.describe_matchup())
env.run(frames=20*30)   # 20s (at 30fps)

# Looks good, nothing seems to be off. Nothing happens after 20s though.
# Need to decide if that's desirable

stage: battlefield
P1: human, character=marth
P2: in-game CPU level 9, character=samus
run(): 600 frames, 0 dropped


{}

In [13]:
# --- Scripted bot (dashdance) vs ZeroBot, 300 frames ------------------------

script = [
    (NOOP, 100),   # warm up past the entrance/countdown freeze
    (DOWN|SPECIAL, 10),
    (LEFT, 20),
    (RIGHT, 20),
    (LEFT, 20),
    (RIGHT, 20),
]

obs, info = env.reset(
    players={
        1: ScriptedBot(Character.Marth, script, on_end="loop"),
        2: ZeroBot(Character.Samus),
    },
)
print(env.describe_matchup())
traj = env.run(frames=150 + 300, record=True)


# works as intended, then reverts back to cpu (!, undesired))

stage: battlefield
P1: Python ScriptedBot (scripted), character=marth
P2: Python ZeroBot (zero), character=samus
run(): 450 frames, 0 dropped


In [14]:
# --- Lockstep stepping: debugger-safe one-frame transitions ------------------
# reset() returns while the game is paused. Set a breakpoint after either
# reset() or step(): the game remains frozen until the next env.step() call.
# This uses the rebuilt RL bridge; restart the game after rebuilding the SWF.

lockstep_env = SSF2Env(lockstep=True)
obs, info = lockstep_env.reset(
    players={
        1: Agent(Character.Marth),
        2: ZeroBot(Character.Samus),
    },
    stage=Stage.bf,
)
assert info["lockstep"] and info["paused"]
frame_before = info["frame"]
print(f"Paused at frame {frame_before}. Set a breakpoint here if desired.")

# The game advances exactly one simulation frame, then pauses again.
obs, reward, terminated, truncated, info = lockstep_env.step(0)  # discrete "noop"
for _ in range(30):
    lockstep_env.step(0)

assert info["paused"]
assert info["frame"] == frame_before + 1
print(f"Paused again at frame {info['frame']} (reward={reward:.3f}).")

# Keep lockstep_env open to step manually. Call lockstep_env.close() when done.


Paused at frame 2. Set a breakpoint here if desired.


BridgeError: timed out waiting for 'step_complete' reply to request 21

In [ ]:
# --- Human vs FollowBot (observation sanity check) --------------------------
# P1 is you; P2 walks toward you and stops inside a deadzone. If Samus
# visibly chases Marth around the stage, the bridge's x-position data (and
# therefore every observation built from it) is trustworthy.

obs, info = env.reset(
    players={1: Human("marth"), 2: FollowBot("samus", deadzone=30.0)},
)
print(env.describe_matchup())
traj = env.run(frames=600, record=True)   # ~20s: run around, get chased

# Works as intended, then p2 reverts back to cpu

stage: finaldestination
P1: human, character=marth
P2: Python FollowBot (follow), character=samus
run(): 600 frames, 0 dropped


NameError: name 'np' is not defined